In [1]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

/home/tommaso/projects/Board-Game-Data-Analytics/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Caching: embeddings & FAISS index

Encoding ~20k descriptions and building the HNSW index takes a few minutes; both are cached to disk after the first run.

- **`data/embeddings.npy`** - embedding matrix, row-aligned with `df`.
- **`data/faiss_index.bin`** - the FAISS `IndexHNSWFlat`.

`df` itself isn't cached - rebuilding it from the source CSVs takes seconds, so it's always regenerated fresh.

Each artifact gets its own file by design. An earlier version stored embeddings as a column in a Parquet file shared with `8_learning_to_rank.ipynb`; whichever notebook ran last would silently overwrite the other's columns, occasionally dropping `embedding` entirely and raising a `KeyError` with no obvious cause. Splitting the cache per artifact removes that failure mode.

Both files are validated against `len(df)` on load and rebuilt automatically if stale. To force a rebuild after changing the model, data, or preprocessing, just delete `embeddings.npy` / `faiss_index.bin`.

In [2]:
DATA_DIR = '../data'
EMBEDDINGS_PATH = os.path.join(DATA_DIR, 'embeddings.npy')
FAISS_INDEX_PATH = os.path.join(DATA_DIR, 'faiss_index.bin')

# df is always rebuilt fresh - merging the CSVs is fast, so there's nothing worth caching here.
df_clean = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_dataset.csv'))
df_raw = pd.read_csv(os.path.join(DATA_DIR, 'DM1_game_dataset.csv'))
df_text = df_raw[['BGGId', 'Description']]
df = pd.merge(df_clean, df_text, on='BGGId', how='inner')
df = df.dropna(subset=['Description']).reset_index(drop=True)

print(f"Dataset ready! Total games: {len(df)}")

Dataset ready! Total games: 19728


## Generate dense embeddings

**Model: `all-MiniLM-L6-v2`.** Chosen for speed and footprint over raw accuracy - CPU-only encoding
of ~20k descriptions finishes in under a minute, and 384-dim vectors keep the FAISS index small and
fast to query. A larger model like `all-mpnet-base-v2` would likely rank slightly better at 2-3x the
compute and storage cost; not worth it for this dataset size and use case.

In [3]:
# Load the pre-trained Neural IR model (needed either way: to encode new queries later)
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = None
if os.path.exists(EMBEDDINGS_PATH):
    cached_embeddings = np.load(EMBEDDINGS_PATH)
    if cached_embeddings.shape[0] == len(df):
        embeddings = cached_embeddings.astype('float32')
        print(f"Embeddings loaded from cache! Shape: {embeddings.shape}")
    else:
        print(
            f"Cache found at '{EMBEDDINGS_PATH}' but row count doesn't match df "
            f"({cached_embeddings.shape[0]} vs {len(df)}) - treating as stale, will recompute."
        )

if embeddings is None:
    descriptions = df['Description'].tolist()

    print("Encoding descriptions into vectors...")
    embeddings = model.encode(descriptions, show_progress_bar=True)
    embeddings = np.array(embeddings).astype('float32')

    print(f"Embeddings generated! Shape: {embeddings.shape}")

    np.save(EMBEDDINGS_PATH, embeddings)
    print(f"Cached embeddings to '{EMBEDDINGS_PATH}'.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15755.98it/s]


Embeddings loaded from cache! Shape: (19728, 384)


## Approximate k-NN with FAISS (HNSW)
Same load-or-build pattern, applied to the index itself: if a cached `faiss_index.bin` exists and
its vector count matches `len(df)`, it's loaded directly with `faiss.read_index()` instead of
rebuilding the HNSW graph from scratch. Otherwise the index is built fresh from `embeddings` and
persisted with `faiss.write_index()` for next time.

Keeping the embedding cache and the index cache as two separate files (rather than one combined
artifact) means a mismatch between them is caught explicitly by the row-count check, instead of
silently searching an index built from stale vectors.

In [4]:
# Get the vector dimension size (384)
dimension = embeddings.shape[1]

index = None
if os.path.exists(FAISS_INDEX_PATH):
    cached_index = faiss.read_index(FAISS_INDEX_PATH)
    if cached_index.ntotal == len(df):
        index = cached_index
        print(f"FAISS index loaded from cache! Total vectors: {index.ntotal}")
    else:
        print(
            f"Cache found at '{FAISS_INDEX_PATH}' but vector count doesn't match df "
            f"({cached_index.ntotal} vs {len(df)}) - treating as stale, will rebuild."
        )

if index is None:
    # Initialize the HNSW index
    # The '32' is the number of connections each node has in the graph
    index = faiss.IndexHNSWFlat(dimension, 32)

    print("Building the HNSW graph...")
    index.add(embeddings)
    print(f"Index built! Total vectors in FAISS: {index.ntotal}")

    faiss.write_index(index, FAISS_INDEX_PATH)
    print(f"Cached FAISS index to '{FAISS_INDEX_PATH}'.")

FAISS index loaded from cache! Total vectors: 19728


## Semantic Query Processing

In [5]:
def search_games_faiss(query, top_k=5):
    """
    Searches the FAISS index using semantic embeddings.
    """
    # 1. Convert the user query into a dense vector
    query_vector = model.encode([query]).astype('float32')

    # 2. Perform the Approximate k-NN search
    # distances are the L2 distance (lower is more similar)
    distances, indices = index.search(query_vector, top_k)

    # 3. Retrieve the matching games
    results = df.iloc[indices[0]].copy()
    results['faiss_distance'] = distances[0]

    return results[['Name', 'faiss_distance', 'Description']]

### Test the neural engine

In [6]:
user_query = "A cooperative space game involving betrayal"
results = search_games_faiss(user_query, top_k=5)

print(f"--- Semantic Results for: '{user_query}' ---\n")
for i, row in results.iterrows():
    print(f"{row['Name']} (Distance: {row['faiss_distance']:.2f})")
    print(f"Snippet: {row['Description'][:150]}...\n")

--- Semantic Results for: 'A cooperative space game involving betrayal' ---

The Crew: The Quest for Planet Nine (Distance: 0.78)
Snippet: cooperative tricktaking game crew quest planet player set astronaut uncertain space adventure rumor unknown planet eventful journey space extend   exc...

Contact: Signals from Outer Space (Distance: 0.85)
Snippet: collaborative space adventure game contact follow signal know exactly come send signal want lead certain planet instead space coordinate receive stran...

Cosmic Encounter (Distance: 0.88)
Snippet: build galactic empirein depth space alien race cosmo vie control universe alliance form shift moment moment cataclysmic battle send starship scream wa...

Star Clicker (Distance: 0.89)
Snippet: star clicker new cooperative game christophe raimbault   author colt express   player kid try save planet evil alien parent mission far far awayldquoe...

Cutthroat Caverns (Distance: 0.89)
Snippet: quotwithout teamwork survive betrayal yoursquoll winquo